# Preprocessing and Splitting

This section converts the cleaned collision dataset into model-ready inputs.

Steps:
- define the target and feature set
- split into train, validation, and test sets
- encode categorical features
- scale numeric features
- fit everything on training only, then transform the rest
- save outputs for reproducibility

### Step 1: Imports and Load the Cleaned Dataset (from data prep)

In [12]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
# Load from the cleaned export
# If df is already in memory from the cleaning section, replace with: df_model = df.copy()
df_model = pd.read_csv('../Data/Cleaned/collisions_cleaned.csv')

print('Loaded shape:', df_model.shape)
df_model.head()

Loaded shape: (8892, 20)


,ACCIDENT_YEAR,ACCIDENT_MONTH,ACCIDENT_DAY,ACCIDENT_HOUR,ACCIDENT_WEEKDAY,ACCIDENTLOCATION,IMPACTLOCATION,INITIALIMPACTTYPE,LIGHT,ROADJURISDICTION,TRAFFICCONTROL,TRAFFICCONTROLCONDITION,THRULANENO,NORTHBOUNDDISOBEYCOUNT,SOUTHBOUNDDISOBEYCOUNT,PEDESTRIANINVOLVED,CYCLISTINVOLVED,MOTORCYCLISTINVOLVED,ENVIRONMENTCONDITION1,injury
0,2015,1,1,0,Thursday,Intersection related,Off highway,SMV other,"Daylight, artificial",Municipal (excl. Twp. Rd.),Stop sign,Functioning,0,0,0,False,False,False,Clear,0
1,2015,1,1,8,Thursday,At/near private drive,Thru lane,SMV unattended vehicle,Daylight,Municipal (excl. Twp. Rd.),No control,unknown,0,0,0,False,False,False,Clear,0
2,2015,1,1,17,Thursday,Other,Not on roadway - left side,SMV other,"Daylight, artificial",Municipal (excl. Twp. Rd.),No control,unknown,0,0,0,False,False,False,Clear,0
3,2015,1,2,22,Friday,At intersection,Within intersection,SMV other,"Dark, artificial",Municipal (excl. Twp. Rd.),Stop sign,Functioning,0,0,0,False,False,False,Clear,0
4,2015,1,3,9,Saturday,Non intersection,Not on roadway - right side,SMV other,Daylight,Regional municipality,No control,unknown,0,0,0,False,False,False,Clear,0


### Step 2: Define the target and feature columns

The target is `injury` (0 = no injury, 1 = injury).

These features are all available when e a collision is recorded and don't directly encode the outcome. `ACCIDENT_YEAR`, `ACCIDENT_MONTH`, and `ACCIDENT_DAY` are left out since they capture time trends rather than collision conditions, so keeping them would let the model learn patterns related to time or specific years instead of those generalizable factors.

In [13]:
target_col = 'injury'

feature_cols = [
    'ACCIDENT_HOUR',
    'ACCIDENT_WEEKDAY',
    'ACCIDENTLOCATION',
    'INITIALIMPACTTYPE',
    'IMPACTLOCATION',
    'LIGHT',
    'ROADJURISDICTION',
    'TRAFFICCONTROL',
    'TRAFFICCONTROLCONDITION',
    'ENVIRONMENTCONDITION1',
    'THRULANENO',
    'NORTHBOUNDDISOBEYCOUNT',
    'SOUTHBOUNDDISOBEYCOUNT',
    'PEDESTRIANINVOLVED',
    'CYCLISTINVOLVED',
    'MOTORCYCLISTINVOLVED'
]

# Only keep features that are actually in the cleaned data
feature_cols = [col for col in feature_cols if col in df_model.columns]

X = df_model[feature_cols].copy()
y = df_model[target_col].copy()

print('X shape:', X.shape)
print('y shape:', y.shape)
print('\nTarget distribution:')
print(y.value_counts())
print(y.value_counts(normalize=True).round(3))

X shape: (8892, 16)
y shape: (8892,)

Target distribution:
injury
0    7681
1    1211
Name: count, dtype: int64
injury
0    0.864
1    0.136
Name: proportion, dtype: float64


## Feature choice Explanation:

The first modeling pass keeps variables that describe:
- time of collision: `ACCIDENT_HOUR`, `ACCIDENT_WEEKDAY`
- crash location and impact context: `ACCIDENTLOCATION`, `IMPACTLOCATION`, `INITIALIMPACTTYPE`
- visibility and surroundings: `LIGHT`, `ENVIRONMENTCONDITION1`
- road and traffic-control context: `ROADJURISDICTION`, `TRAFFICCONTROL`, `TRAFFICCONTROLCONDITION`
- roadway structure and movement counts: `THRULANENO`, `NORTHBOUNDDISOBEYCOUNT`, `SOUTHBOUNDDISOBEYCOUNT`
- vulnerable road user involvement: `PEDESTRIANINVOLVED`, `CYCLISTINVOLVED`, `MOTORCYCLISTINVOLVED`

These features were kept because they're available before model training and they don't directly encode the target. Also, they showed meaningful patterns in the cleaning and EDA stages. Any stricter feature selection will be done during model development using training and validation data only.

### Step 3: Check for Missing Values and Fix the boolean flag columns

Double checking that the selected modelling columns are clean before splitting.

The vulnerable user columns store the strings `'True'` and `'False'` instead of actual booleans. We convert them to `'yes'` / `'no'` before anything else so the encoder treats them as proper nominal categories.

In [14]:
print('Missing values in selected features and target:')
print(df_model[feature_cols + [target_col]].isna().sum())
flag_cols = ['PEDESTRIANINVOLVED', 'CYCLISTINVOLVED', 'MOTORCYCLISTINVOLVED']
flag_cols = [col for col in flag_cols if col in X.columns]

for col in flag_cols:
    X[col] = X[col].astype(str).str.strip().str.lower()
    X[col] = X[col].replace({
        'true':  'yes',
        'false': 'no',
        '1':     'yes',
        '0':     'no',
        'y':     'yes',
        'n':     'no'
    })

# Check the values after standardizing
for col in flag_cols:
    print(f'\n{col}:')
    print(X[col].value_counts(dropna=False))

Missing values in selected features and target:
ACCIDENT_HOUR              0
ACCIDENT_WEEKDAY           0
ACCIDENTLOCATION           0
INITIALIMPACTTYPE          0
IMPACTLOCATION             0
LIGHT                      0
ROADJURISDICTION           0
TRAFFICCONTROL             0
TRAFFICCONTROLCONDITION    0
ENVIRONMENTCONDITION1      0
THRULANENO                 0
NORTHBOUNDDISOBEYCOUNT     0
SOUTHBOUNDDISOBEYCOUNT     0
PEDESTRIANINVOLVED         0
CYCLISTINVOLVED            0
MOTORCYCLISTINVOLVED       0
injury                     0
dtype: int64

PEDESTRIANINVOLVED:
PEDESTRIANINVOLVED
no     8655
yes     237
Name: count, dtype: int64

CYCLISTINVOLVED:
CYCLISTINVOLVED
no     8731
yes     161
Name: count, dtype: int64

MOTORCYCLISTINVOLVED:
MOTORCYCLISTINVOLVED
no     8828
yes      64
Name: count, dtype: int64


### Step 4: Vaulnerable User Feature Engineering and Separate numeric and categorical columns

We're creating a new column called `VULNERABLE_USER_INVOLVED` to flag whether any pedestrian, cyclist, or motorcyclist was involved in the collision. Based on the EDA, those three variables showed the clearest relationship with injury outcome. The three individual columns are kept alongside this new one so the model can still use the more specific signals.

Nominal categorical columns get one-hot encoded instead of label encoding since we want to avoid the implied ordering that comes from assigning integers.

In [15]:
X['VULNERABLE_USER_INVOLVED'] = np.where(
    (X['PEDESTRIANINVOLVED'] == 'yes') |
    (X['CYCLISTINVOLVED']    == 'yes') |
    (X['MOTORCYCLISTINVOLVED'] == 'yes'),
    'yes',
    'no'
)

print('VULNERABLE_USER_INVOLVED:')
print(X['VULNERABLE_USER_INVOLVED'].value_counts(dropna=False))

numeric_features = [
    'ACCIDENT_HOUR',
    'THRULANENO',
    'NORTHBOUNDDISOBEYCOUNT',
    'SOUTHBOUNDDISOBEYCOUNT'
]

categorical_features = [
    'ACCIDENT_WEEKDAY',
    'ACCIDENTLOCATION',
    'INITIALIMPACTTYPE',
    'IMPACTLOCATION',
    'LIGHT',
    'ROADJURISDICTION',
    'TRAFFICCONTROL',
    'TRAFFICCONTROLCONDITION',
    'ENVIRONMENTCONDITION1',
    'PEDESTRIANINVOLVED',
    'CYCLISTINVOLVED',
    'MOTORCYCLISTINVOLVED',
    'VULNERABLE_USER_INVOLVED'
]

# Only include columns that actually ended up in X
numeric_features     = [col for col in numeric_features     if col in X.columns]
categorical_features = [col for col in categorical_features if col in X.columns]

print('Numeric features:    ', numeric_features)
print('Categorical features:', categorical_features)

VULNERABLE_USER_INVOLVED:
VULNERABLE_USER_INVOLVED
no     8436
yes     456
Name: count, dtype: int64
Numeric features:     ['ACCIDENT_HOUR', 'THRULANENO', 'NORTHBOUNDDISOBEYCOUNT', 'SOUTHBOUNDDISOBEYCOUNT']
Categorical features: ['ACCIDENT_WEEKDAY', 'ACCIDENTLOCATION', 'INITIALIMPACTTYPE', 'IMPACTLOCATION', 'LIGHT', 'ROADJURISDICTION', 'TRAFFICCONTROL', 'TRAFFICCONTROLCONDITION', 'ENVIRONMENTCONDITION1', 'PEDESTRIANINVOLVED', 'CYCLISTINVOLVED', 'MOTORCYCLISTINVOLVED', 'VULNERABLE_USER_INVOLVED']


### Step 5: Train / validation / test split

There are 3 main things we need to do here:

-Train:fits the model parameters

-Validation: used during model development to tune hyperparameters and compare models

-Test: held out completely until the very end

We use a 60 / 20 / 20 split. `stratify=y` keeps the injury class proportion the same across all three sets, which matters because the target is imbalanced (determined during EDA).

In [16]:
# First cut: hold out 20% for the test set
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    shuffle=True,
    stratify=y
)

# Second cut: from the remaining 80%, hold out 25% for validation
# 0.25 * 0.80 = 0.20 of the total, so the final split is 60 / 20 / 20
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.25,
    random_state=42,
    shuffle=True,
    stratify=y_train_full
)

print('Train:     ', X_train.shape)
print('Validation:', X_val.shape)
print('Test:      ', X_test.shape)
# Check that injury rate is consistent across splits
print('Injury rate by split:')
print('Train:     ', round(y_train.mean(), 3))
print('Validation:', round(y_val.mean(), 3))
print('Test:      ', round(y_test.mean(), 3))

Train:      (5334, 17)
Validation: (1779, 17)
Test:       (1779, 17)
Injury rate by split:
Train:      0.136
Validation: 0.136
Test:       0.136


## Note: class imbalance

The injury target is imbalanced as non-injury collisions make up the majority of the data. Stratified splitting is used here to preserve the class proportions across train, validation, and test.

The model level handling of the imbalance (class weights, evaluation metrics, etc) will be decided during model training and validation.

### Step 6: Build the preprocessing pipeline

`ColumnTransformer` applies different transformations to different columns at once. Two transformers:

- `StandardScaler` on numeric columns: subtracts the mean and divides by std, so all numeric inputs are on the same scale
- `OneHotEncoder` on categorical columns: converts each category value to a binary column

`handle_unknown='ignore'` means if a category shows up in validation or test but not in training, it gets encoded as all zeros instead of raising an error.

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='drop'
)

### Step 7: Fit on training only, then transform everything

`fit_transform` on the training set learns the scaler parameters (mean, std) and the encoder categories from training rows only.

`transform` on validation and test applies those same learned parameters without looking at the held out data again. If we ran `fit_transform` on all the data first, the scaler would see the test rows during fitting, which leaks future information so not the method we should use.

In [18]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed   = preprocessor.transform(X_val)
X_test_processed  = preprocessor.transform(X_test)

print('Processed train:     ', X_train_processed.shape)
print('Processed validation:', X_val_processed.shape)
print('Processed test:      ', X_test_processed.shape)

Processed train:      (5334, 89)
Processed validation: (1779, 89)
Processed test:       (1779, 89)


### Step 8: Inspect the processed feature names

After one-hot encoding, each category value becomes its own binary column. This shows exactly what the model will see.

In [19]:
feature_names = preprocessor.get_feature_names_out()

print('Total processed features:', len(feature_names))
print('\nFirst 30 feature names:')
print(feature_names[:30])

Total processed features: 89

First 30 feature names:
['num__ACCIDENT_HOUR' 'num__THRULANENO' 'num__NORTHBOUNDDISOBEYCOUNT'
 'num__SOUTHBOUNDDISOBEYCOUNT' 'cat__ACCIDENT_WEEKDAY_Friday'
 'cat__ACCIDENT_WEEKDAY_Monday' 'cat__ACCIDENT_WEEKDAY_Saturday'
 'cat__ACCIDENT_WEEKDAY_Sunday' 'cat__ACCIDENT_WEEKDAY_Thursday'
 'cat__ACCIDENT_WEEKDAY_Tuesday' 'cat__ACCIDENT_WEEKDAY_Wednesday'
 'cat__ACCIDENTLOCATION_At intersection'
 'cat__ACCIDENTLOCATION_At railway crossing'
 'cat__ACCIDENTLOCATION_At/near private drive'
 'cat__ACCIDENTLOCATION_Intersection related'
 'cat__ACCIDENTLOCATION_Non intersection' 'cat__ACCIDENTLOCATION_Other'
 'cat__ACCIDENTLOCATION_Overpass or bridge'
 'cat__ACCIDENTLOCATION_Underpass or tunnel'
 'cat__INITIALIMPACTTYPE_Angle' 'cat__INITIALIMPACTTYPE_Approaching'
 'cat__INITIALIMPACTTYPE_Other' 'cat__INITIALIMPACTTYPE_Rear end'
 'cat__INITIALIMPACTTYPE_SMV other'
 'cat__INITIALIMPACTTYPE_SMV unattended vehicle'
 'cat__INITIALIMPACTTYPE_Sideswipe'
 'cat__INITIALIMPACTT

### Step 9: Checks, Split Summary, and Saving

In [20]:
assert X_train_processed.shape[0] == len(y_train), 'Train size mismatch'
assert X_val_processed.shape[0]   == len(y_val),   'Validation size mismatch'
assert X_test_processed.shape[0]  == len(y_test),  'Test size mismatch'

print('All checks passed.')

All checks passed.


In [21]:
split_summary = pd.DataFrame({
    'split':       ['train', 'validation', 'test'],
    'rows':        [len(y_train), len(y_val), len(y_test)],
    'injury_rate': [round(y_train.mean(), 3), round(y_val.mean(), 3), round(y_test.mean(), 3)]
})

split_summary

,split,rows,injury_rate
0,train,5334,0.136
1,validation,1779,0.136
2,test,1779,0.136


In [22]:
output_dir = '../Data/Processed'
os.makedirs(output_dir, exist_ok=True)

np.save(f'{output_dir}/X_train_processed.npy', X_train_processed)
np.save(f'{output_dir}/X_val_processed.npy',   X_val_processed)
np.save(f'{output_dir}/X_test_processed.npy',  X_test_processed)

y_train.to_csv(f'{output_dir}/y_train.csv', index=False)
y_val.to_csv(f'{output_dir}/y_val.csv',     index=False)
y_test.to_csv(f'{output_dir}/y_test.csv',   index=False)

pd.Series(feature_names, name='feature_name').to_csv(
    f'{output_dir}/feature_names.csv', index=False
)

split_summary.to_csv(f'{output_dir}/split_summary.csv', index=False)

print('Saved to', output_dir)

Saved to ../Data/Processed


## Summary:
At this point:
- feature set and target are fixed
- 60 / 20 / 20 split with stratification on injury class
- nominal categories are one-hot encoded
- numeric features are standardized
- the preprocessor was fit on training only
- outputs are saved and ready for model training